In [1]:
!pip install keras-facenet

import numpy as np
import cv2
import tensorflow as tf
from keras_facenet import FaceNet
from google.colab import drive,files
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img, img_to_array

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.6 MB/s eta 0:00:00
  Created wheel for keras-facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10367 sha256=9ac5bae225b542d35fb7b0949fc5df1774380c608a64fbd0fa48cadf29a6f0ad
  Stored in directory: /root/.cache/pip/wheels/05/b0/f5/19ac49fedc10b1df3ee56b096edbcfa39d45794fccc6bcdbbf
Successfully built keras-facenet


In [2]:
from datasets import load_dataset

ds = load_dataset("tonyassi/celebrity-1000")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18184 [00:00<?, ? examples/s]

In [3]:
drive.mount('/content/drive')
embedder = FaceNet()
model = tf.keras.models.load_model('/content/drive/My Drive/classifier1K.h5')

Mounted at /content/drive


In [4]:
def resize_to_160(img):
    """
    img: Tensor or numpy array of shape (256, 256, 3)
    returns: Tensor (160, 160, 3)
    """
    img = tf.image.resize(img, (160, 160), method='bilinear')
    img = img.numpy()
    return img

In [5]:
def get_embedding(img):
  """
  The embedder.embeddings function expects a list of numpy arrays.
  img is already a numpy array from resize_to_160, so we wrap it in a list.
  """
  img = resize_to_160(img)
  emb = embedder.embeddings([img])[0]
  #emb = np.expand_dims(emb, axis=0)
  return emb

In [6]:
input_layer = tf.keras.Input(shape=(160, 160, 3))
preprocessing = tf.keras.layers.Lambda(lambda x: (x - 127.5) / 127.5)(input_layer)
facenet_base = embedder.model
facenet_base.trainable = False
embedding = facenet_base(preprocessing)
final_output = model(embedding)
full_model = tf.keras.Model(inputs=input_layer, outputs=final_output, name="End_to_End_Corrected")
full_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [7]:
def get_prediction(img):
  emb = get_embedding(img)
  emb = np.expand_dims(emb, axis=0)
  predictions = model.predict(emb)
  return predictions

In [8]:
def get_occlusion_heatmap(full_model, img, true_label_index, patch_size=20, step=5):
    if img.shape[0] != 160:
        img = cv2.resize(img, (160, 160))

    input_img = np.expand_dims(img, axis=0)
    baseline_pred = full_model.predict(input_img, verbose=0)[0]
    baseline_score = baseline_pred[true_label_index]

    heatmap = np.zeros((160, 160))
    h, w, _ = img.shape


    for y in range(0, h - patch_size, step):
        for x in range(0, w - patch_size, step):
            occluded_img = img.copy()
            occluded_img[y:y+patch_size, x:x+patch_size] = 127
            input_occ = np.expand_dims(occluded_img, axis=0)
            occ_pred = full_model.predict(input_occ, verbose=0)[0]

            score = occ_pred[true_label_index]

            importance = baseline_score - score
            heatmap[y:y+patch_size, x:x+patch_size] = importance

    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    return heatmap

In [9]:
def visualize_occlusion_heatmap(label, img):


  test_img = np.array(img)

  heatmap = get_occlusion_heatmap(full_model, test_img, label, patch_size=30, step=10)

  plt.figure(figsize=(10, 5))



  plt.subplot(1, 2, 1)
  plt.imshow(test_img)
  heatmap_resized = cv2.resize(heatmap, (test_img.shape[1], test_img.shape[0]))

  plt.imshow(heatmap_resized, cmap='jet', alpha=0.5)
  plt.title("Importance Heatmap (Red = Crucial)")
  plt.axis('off')

  plt.tight_layout()
  plt.show()

In [10]:
def make_gradcam_heatmap(img_array, facenet_model, classifier_model):
    last_conv_layer_name = None

    for layer in reversed(facenet_model.layers):
        if 'Conv2D' in layer.__class__.__name__:
            last_conv_layer_name = layer.name
            print(f"Success! Targeting Conv Layer: {layer.name}")
            break

    if not last_conv_layer_name:
        print("Error: No Conv2D layer found. Here are the last 5 layers:")
        for layer in facenet_model.layers[-5:]:
            print(f"- {layer.name} ({layer.__class__.__name__})")
        return None

    grad_model = tf.keras.Model(
        inputs=facenet_model.inputs,
        outputs=[facenet_model.get_layer(last_conv_layer_name).output, facenet_model.output]
    )

    with tf.GradientTape() as tape:
        preprocessed_img = (tf.cast(img_array, tf.float32) - 127.5) / 127.5

        conv_outputs, embeddings = grad_model(preprocessed_img)
        preds = classifier_model(embeddings)

        top_pred_index = tf.argmax(preds[0])
        loss = preds[:, top_pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

In [11]:
def grad_cam(img):
  test_img = np.array(img)

  if test_img.shape[0] != 160:
      test_img = cv2.resize(test_img, (160, 160))
  img_batch = np.expand_dims(test_img, axis=0)

  heatmap = make_gradcam_heatmap(img_batch, embedder.model, model)

  if heatmap is not None:
      plt.figure(figsize=(12, 6))


      plt.subplot(1, 1, 1)
      plt.imshow(test_img)
      heatmap_resized = cv2.resize(heatmap, (160, 160))
      plt.imshow(heatmap_resized, cmap='jet', alpha=0.6)
      plt.title("Grad-CAM Heatmap\n(Where the model is looking)")
      plt.axis('off')

      plt.tight_layout()
      plt.show()

In [12]:
from mtcnn import MTCNN
import numpy as np
import cv2

detector = MTCNN()

def crop_face_mtcnn(img, required_size=(160, 160)):
    """
    Detects a face using MTCNN and returns a cropped RGB face.
    The input MUST be an RGB uint8 numpy array.
    """

    results = detector.detect_faces(img)
    if len(results) == 0:
        return None

    x, y, w, h = results[0]['box']

    x = max(0, x)
    y = max(0, y)
    x2 = min(x + w, img.shape[1])
    y2 = min(y + h, img.shape[0])

    face = img[y:y2, x:x2]

    if face.size == 0:
        return None

    face = cv2.resize(face, required_size)

    return face


In [13]:
!wget https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml

--2025-12-22 07:18:11--  https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 930127 (908K) [text/plain]
Saving to: ‘haarcascade_frontalface_default.xml’

haarcascade_frontal 100%[===================>] 908.33K  --.-KB/s    in 0.03s   

2025-12-22 07:18:11 (26.1 MB/s) - ‘haarcascade_frontalface_default.xml’ saved [930127/930127]



In [15]:
import gradio as gr
import cv2
import numpy as np
import tensorflow as tf
from mtcnn import MTCNN

try:
    class_names = ds['train'].features['label'].names
except:
    class_names = [str(i) for i in range(10)]
haar_detector = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

detector = MTCNN()

def analyze_upload(input_img, enable_occlusion):
    if input_img is None:
        return None, None, None, "Please upload an image."

    img = np.array(input_img).astype('uint8')

    cropped = crop_face_mtcnn(img)

    if cropped is None:
        return None, None, None, "❌ No face detected."

    cropped_resized = cv2.resize(cropped, (160, 160))
    img_batch = np.expand_dims(cropped_resized, axis=0)

    predictions = full_model.predict(img_batch, verbose=0)[0]
    predicted_class = np.argmax(predictions)
    confidence = np.max(predictions) * 100
    name = class_names[predicted_class]

    top_indices = np.argsort(predictions)[::-1][:10]
    stats_text = f"🏆 Predicted: {name}\nConfident: {confidence:.2f}%\n\n📊 Top 10 Matches:\n"
    for idx in top_indices:
        stats_text += f"{class_names[idx]}: {predictions[idx]*100:.2f}%\n"

    gradcam_display = None
    try:
        heatmap_grad = make_gradcam_heatmap(img_batch, embedder.model, model)
        if heatmap_grad is not None:
            heatmap_grad_resized = cv2.resize(heatmap_grad, (160, 160))
            heatmap_grad_colored = cv2.applyColorMap(np.uint8(255 * heatmap_grad_resized), cv2.COLORMAP_JET)
            heatmap_grad_colored = cv2.cvtColor(heatmap_grad_colored, cv2.COLOR_BGR2RGB)
            gradcam_display = cv2.addWeighted(cropped_resized, 0.6, heatmap_grad_colored, 0.4, 0)
    except Exception as e:
        print(f"Grad-CAM Error: {e}")
        gradcam_display = cropped_resized

    occlusion_display = None
    if enable_occlusion:
        try:
            heatmap_occ = get_occlusion_heatmap(full_model, cropped_resized, predicted_class, patch_size=30, step=10)

            heatmap_occ_resized = cv2.resize(heatmap_occ, (160, 160))
            heatmap_occ_colored = cv2.applyColorMap(np.uint8(255 * heatmap_occ_resized), cv2.COLORMAP_JET)
            heatmap_occ_colored = cv2.cvtColor(heatmap_occ_colored, cv2.COLOR_BGR2RGB)
            occlusion_display = cv2.addWeighted(cropped_resized, 0.6, heatmap_occ_colored, 0.4, 0)
        except Exception as e:
            print(f"Occlusion Error: {e}")
            occlusion_display = cropped_resized
    else:
        occlusion_display = cropped_resized

    return cropped, gradcam_display, occlusion_display, stats_text


def live_prediction(image):
    if image is None:
        return None

    img_np = np.array(image)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    faces = haar_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=8, minSize=(60, 60))

    for (x, y, w, h) in faces:
        face_crop = img_np[y:y+h, x:x+w]

        if face_crop.shape[0] < 20 or face_crop.shape[1] < 20:
            continue

        try:
            face_resized = cv2.resize(face_crop, (160, 160))
            img_batch = np.expand_dims(face_resized, axis=0)

            preds = full_model.predict(img_batch, verbose=0)[0]
            best_idx = np.argmax(preds)
            conf = np.max(preds) * 100

            if conf < 50:
                continue

            name = class_names[best_idx]

            color = (0, 255, 0) if conf > 75 else (255, 165, 0)
            cv2.rectangle(img_np, (x, y), (x+w, y+h), color, 2)

            cv2.rectangle(img_np, (x, y-25), (x+w, y), color, -1)
            cv2.putText(img_np, f"{name} {conf:.0f}%", (x+5, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)

        except:
            pass

    return img_np



with gr.Blocks(title="Celebrity Recognition Suite") as demo:
    gr.Markdown("# Celebrity Face Recognition")

    with gr.Tab("🖼️ Image Analysis"):
        gr.Markdown("Upload an image. Enable **Occlusion Heatmap** only if you need deep debugging (it is slow).")

        with gr.Row():
            with gr.Column():
                input_image = gr.Image(sources=["upload"], label="Upload Image")

                check_occlusion = gr.Checkbox(label="Enable Occlusion Heatmap (Slow)", value=False)

                btn_analyze = gr.Button("Analyze Face", variant="primary")

            with gr.Column():
                stats_box = gr.Textbox(label="Prediction Stats", lines=5)

        with gr.Row():
            out_crop = gr.Image(label="Cropped Face")
            out_gradcam = gr.Image(label="Grad-CAM (Attention)")
            out_occlusion = gr.Image(label="Occlusion (Sensitivity)")

        btn_analyze.click(
            fn=analyze_upload,
            inputs=[input_image, check_occlusion],
            outputs=[out_crop, out_gradcam, out_occlusion, stats_box]
        )

    with gr.Tab("📹 Live Webcam"):
        gr.Markdown("Click **Record** (or Start) to begin the stream.")

        with gr.Row():
            webcam_input = gr.Image(sources=["webcam"], streaming=True, label="Input")
            webcam_output = gr.Image(label="Output")

        webcam_input.stream(fn=live_prediction, inputs=webcam_input, outputs=webcam_output)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7c441f569d45866276.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",

Success! Targeting Conv Layer: Block8_6_Conv2d_1x1


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",

Success! Targeting Conv Layer: Block8_6_Conv2d_1x1
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7c441f569d45866276.gradio.live
